# Mathula TV Pyannote analysis worker
Run every cell from top to bottom in a Colab T4 GPU runtime. GCS is authoritative and no credentials are stored in this notebook.

In [ ]:
%pip install -q --upgrade google-cloud-storage
print('Bootstrap dependencies installed')

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Google authentication complete')

In [ ]:
PROJECT_ID = "bakgethwa"
BUCKET = "bakgethwa-mathula-tv"
PREFIX = "mathula-tv"
JOB_ID = "19ba6d69f1b84132ba4f20599101834a"
FORCE = False
LEASE_SECONDS = 900
print({'project': PROJECT_ID, 'bucket': BUCKET, 'prefix': PREFIX, 'job_id': JOB_ID, 'force': FORCE})

In [ ]:
import hashlib, json, pathlib, subprocess, sys, tempfile
from google.cloud import storage
client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET)
assert bucket.exists(), f'Cannot access gs://{BUCKET}'
manifest_blob = bucket.blob(f'{PREFIX}/runtime/manifest.json')
assert manifest_blob.exists(), 'Runtime package is missing; run publish-colab-package on the server'
package_manifest = json.loads(manifest_blob.download_as_text())
wheel_path = pathlib.Path(tempfile.gettempdir()) / pathlib.Path(package_manifest['wheel_object']).name
bucket.blob(package_manifest['wheel_object']).download_to_filename(str(wheel_path))
actual_hash = hashlib.sha256(wheel_path.read_bytes()).hexdigest()
assert actual_hash == package_manifest['sha256'], 'Downloaded wheel SHA-256 mismatch'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', str(wheel_path)], check=True)
print({'package_version': package_manifest['package_version'], 'wheel_verified': True})

In [ ]:
%pip install -q --upgrade 'pyannote.audio>=3.1'
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU'
print({'cuda': True, 'device': torch.cuda.get_device_name(0)})

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get("HUGGINGFACE_TOKEN")
if not HF_TOKEN:
    raise RuntimeError('Add HUGGINGFACE_TOKEN in Colab Secrets and grant this notebook access')
print('Hugging Face secret is available')

In [ ]:
from mathula_tv.colab_worker import discover_eligible_jobs, eligible_job
from mathula_tv.gcs_store import GCSStore
store = GCSStore(BUCKET, PREFIX, client=client)
eligible = discover_eligible_jobs(client, BUCKET, PREFIX)
assert JOB_ID in eligible or FORCE, f'Configured job is not eligible. Eligible jobs: {eligible}'
status, _ = store.download_json(JOB_ID, 'status.json')
result_exists = store.bucket.blob(store.name(JOB_ID, 'analysis/pyannote_diarization.json')).exists()
eligible_job(status, force=FORCE, result_exists=result_exists)
print({'eligible_jobs': eligible, 'selected_job': JOB_ID})

In [ ]:
import pathlib, tempfile, uuid
from mathula_tv.colab_worker import run_analysis_worker
WORKER_ID = f'colab-{uuid.uuid4().hex[:12]}'
result = run_analysis_worker(store, JOB_ID, HF_TOKEN, pathlib.Path(tempfile.gettempdir()) / 'mathula-tv', worker_id=WORKER_ID, force=FORCE, lease_seconds=LEASE_SECONDS)
print('Pyannote analysis and generation-safe upload completed')

In [ ]:
result_blob = store.bucket.blob(store.name(JOB_ID, 'analysis/pyannote_diarization.json'))
result_blob.reload()
final_status, _ = store.download_json(JOB_ID, 'status.json')
assert 'pyannote_analysis' in final_status['completed_stages']
print({'job_id': JOB_ID, 'state': final_status['state'], 'stage': 'pyannote_analysis', 'model': result['model_identifier'], 'device': result['device'], 'speakers': result['speaker_count'], 'turns': len(result['turns']), 'processing_seconds': round(result['processing_duration'], 2), 'real_time_factor': round(result['real_time_factor'], 3), 'uploaded_bytes': result_blob.size, 'next': 'Run the server process command for reconciliation'})